# Tree-Based Models, From One Tree to Boosting
### Week 4, Session 1 — concepts

Last week you fitted straight lines. A straight line assumes the effect of a feature is
the same everywhere: one more year of age costs the same whether you are 20 or 60.
Real data is rarely that polite.

Today we build the other big family of models: **trees**. A tree asks yes/no questions.
That lets it carve the feature space into regions and give each region its own answer,
which means it can represent "BMI barely matters for non-smokers but matters a lot for
smokers" **without anyone telling it that interaction exists**.

By the end of this session you can explain, in plain English and with a picture:

1. how a single decision tree picks its questions, and why it overfits
2. why averaging many noisy trees (a **random forest**) beats one careful tree
3. how **boosting** is a completely different idea to bagging, and why it usually wins
4. which of the three to reach for, and when a linear model is still the right answer

**Session plan**

| Phase | What we do | Time |
|---|---|---|
| 1 | One decision tree: splits, impurity, overfitting | 35 min |
| 2 | Bagging and random forests: averaging away variance | 35 min |
| 3 | Boosting: learning from your own mistakes | 35 min |
| 4 | Which model, when | 15 min |

We use the same dataset as this afternoon's project — the **CDC Diabetes Health
Indicators** survey — so that everything you see here is immediately reusable.

---
## Setup

Runs as-is in Google Colab. Locally you need `pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, BaggingClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, accuracy_score

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Ready.')

## The data

**CDC Diabetes Health Indicators** — 253,680 responses to the 2015 Behavioral Risk Factor
Surveillance System survey, 21 health and lifestyle features, and a binary label:
does this person have diabetes (or prediabetes)?

We pull it straight from the UCI repository so nobody needs a login.

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/891/data.csv'

def load_diabetes_data():
    '''UCI first, local file as fallback.'''
    try:
        df = pd.read_csv(DATA_URL)
        print(f'Loaded {len(df):,} rows from UCI.')
    except Exception as e:
        print(f'UCI fetch failed ({e}). Falling back to local file.')
        df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
        print(f'Loaded {len(df):,} rows from local CSV.')
    return df.drop(columns=['ID'], errors='ignore')

df = load_diabetes_data()
df.head()

For the concept session we work on a **20,000-row stratified sample**. Every idea below
shows up identically at full size, and the small sample means each cell finishes in
seconds instead of minutes. You will use the full dataset in the project.

In [ ]:
X_full = df.drop(columns=['Diabetes_binary'])
y_full = df['Diabetes_binary'].astype(int)

X, _, y, _ = train_test_split(
    X_full, y_full, train_size=20_000, stratify=y_full, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print('Sample:', X.shape)
print('Positive rate:', f'{y.mean():.1%}')
print('Train / test:', X_train.shape[0], '/', X_test.shape[0])

> **Instructor note.** Point at that 13.9% positive rate now and keep pointing at it.
> Every model today can score 86% accuracy by answering "no diabetes" to everything.
> We use **ROC-AUC** as the scoreboard for this session, which is not fooled by that
> trick. The project session goes much deeper into choosing the right metric.

---
# Phase 1 — One decision tree (35 min)

## What a tree actually is

A decision tree is a sequence of yes/no questions about one feature at a time.

```
              is HighBP > 0.5 ?
                /            \
              no              yes
             /                  \
   is GenHlth > 2.5 ?      is GenHlth > 2.5 ?
     /       \                /        \
  4% risk   11% risk      12% risk   32% risk
```

That is the whole model. To predict, you walk down the questions until you fall off the
bottom into a **leaf**, and the leaf hands you the answer — for classification, the
proportion of training rows in that leaf that were positive.

No coefficients, no distance, no scaling. Which is our first practical consequence:

> **Trees do not care about feature scale.** Splitting on `BMI > 29.5` gives the exact
> same tree whether BMI is in kg/m² or standardised to mean 0. This is a genuine
> difference from linear models and KNN, and it is one of the most useful facts about
> trees you will use all course.

## How does it choose the question?

Greedily. At each node the tree tries **every feature at every possible threshold**,
scores each candidate split, and keeps the single best one. Then it recurses.

"Best" means: the two children are **purer** than the parent. The default purity
measure for classification is **Gini impurity**:

$$\text{Gini} = 1 - \sum_k p_k^2$$

where $p_k$ is the fraction of the node belonging to class $k$. Zero means the node is
all one class (perfect). 0.5 means a 50/50 coin flip (useless).

Let's compute it by hand, once, so the number stops being a mystery.

In [ ]:
def gini(y_node):
    '''Gini impurity of a set of labels.'''
    if len(y_node) == 0:
        return 0.0
    p = np.bincount(y_node, minlength=2) / len(y_node)
    return 1 - (p ** 2).sum()

def split_gain(X_col, y_vals, threshold):
    '''How much impurity does this split remove? Bigger is better.'''
    left  = y_vals[X_col <= threshold]
    right = y_vals[X_col >  threshold]
    n = len(y_vals)
    weighted_child = (len(left) / n) * gini(left) + (len(right) / n) * gini(right)
    return gini(y_vals) - weighted_child

y_arr = y_train.to_numpy()
print(f'Impurity of the whole training set: {gini(y_arr):.4f}')
print()
print(f'{"candidate split":<28}{"gain":>8}')
for col, thr in [('GenHlth', 3.5), ('BMI', 29.5), ('HighBP', 0.5),
                 ('Age', 7.5), ('Sex', 0.5), ('Fruits', 0.5)]:
    g = split_gain(X_train[col].to_numpy(), y_arr, thr)
    print(f'{col + " > " + str(thr):<28}{g:>8.4f}')

The ordering in that table *is* the tree's decision rule. It would pick the largest gain.
`Fruits` barely moves the number — eating fruit does not separate these groups — and a
greedy tree will simply never ask about it while better questions remain.

**Gini vs entropy:** entropy ($-\sum p_k \log_2 p_k$) is the alternative and it almost
never changes the resulting tree in a way you can measure. Gini is the default because
it avoids a logarithm. Do not spend tuning budget on `criterion`.

## Now let scikit-learn do it

In [ ]:
tree_small = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree_small.fit(X_train, y_train)

plt.figure(figsize=(20, 8))
plot_tree(tree_small,
          feature_names=X_train.columns,
          class_names=['no diabetes', 'diabetes'],
          filled=True, rounded=True, fontsize=9, impurity=True, proportion=True)
plt.title('A depth-3 decision tree on the diabetes data')
plt.show()

In [ ]:
# The same tree as text — often easier to read out loud to a room.
print(export_text(tree_small, feature_names=list(X_train.columns), decimals=2))

**Every single leaf says `class: 0`.** That is not a bug, it is the 13.9% positive rate
biting: no leaf in a depth-3 tree contains a majority of diabetics, so at the default
0.5 threshold everything rounds down to "no".

What actually differs between leaves is the **probability**, and that is the number we
will use all course. Here it is.

In [ ]:
leaf_of_row = tree_small.apply(X_train)
leaf_summary = (pd.DataFrame({'leaf': leaf_of_row, 'diabetes': y_train.to_numpy()})
                .groupby('leaf')
                .agg(people=('diabetes', 'size'), diabetes_rate=('diabetes', 'mean'))
                .sort_values('diabetes_rate', ascending=False))
leaf_summary['diabetes_rate'] = (leaf_summary['diabetes_rate'] * 100).round(1)
leaf_summary

> **Instructor note.** Walk one path out loud: "high blood pressure, general health
> rated 3 or worse, BMI above 27.5 — roughly a third of people like that in our training
> data had diabetes, against 14% in the population." That sentence is the model. Nobody needs a statistics background to audit it, and that
> transparency is the single biggest reason trees survive in regulated industries.

## The decision boundary — what "carving up space" looks like

Two features only (BMI and Age), so we can draw it. Watch what happens as we let the
tree get deeper, and compare against logistic regression, which can only ever draw one
straight line.

In [ ]:
pair = ['BMI', 'Age']
Xp_train, Xp_test = X_train[pair], X_test[pair]

xx, yy = np.meshgrid(np.linspace(12, 55, 300), np.linspace(1, 13, 300))
grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=pair)

models = {
    'Logistic regression': LogisticRegression(max_iter=1000),
    'Tree, max_depth=2': DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
    'Tree, max_depth=5': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Tree, no limit': DecisionTreeClassifier(random_state=RANDOM_STATE),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
for ax, (name, mdl) in zip(axes, models.items()):
    mdl.fit(Xp_train, y_train)
    zz = mdl.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.85)
    train_auc = roc_auc_score(y_train, mdl.predict_proba(Xp_train)[:, 1])
    test_auc  = roc_auc_score(y_test,  mdl.predict_proba(Xp_test)[:, 1])
    ax.set_title(f'{name}\ntrain AUC {train_auc:.3f} | test AUC {test_auc:.3f}')
    ax.set_xlabel('BMI')
axes[0].set_ylabel('Age band')
plt.tight_layout(); plt.show()

**Read those four pictures carefully — this is the core of Phase 1.**

- Logistic regression gives one smooth diagonal. Clean, but it cannot bend.
- Depth 2 gives four rectangles. Crude, but already shaped like the data.
- Depth 5 gives a sensible patchwork.
- Unlimited depth gives *confetti* — tiny islands of certainty built around individual
  survey respondents. Look at the two AUC numbers underneath it.

That last panel is **overfitting**, drawn. The tree has not learned anything about
diabetes; it has memorised 15,000 people.

> **Trees always split on one feature at a time, so every boundary is an axis-aligned
> step.** A tree approximates a diagonal with a staircase. That is the price of the
> flexibility, and it is exactly why ensembles (many staircases, averaged) help so much.

## Overfitting, measured instead of drawn

Back to all 21 features. Sweep `max_depth` and score train vs test.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None]
rows = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({
        'max_depth': 'none' if d is None else d,
        'leaves': t.get_n_leaves(),
        'train AUC': roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]),
        'test AUC':  roc_auc_score(y_test,  t.predict_proba(X_test)[:, 1]),
    })
depth_table = pd.DataFrame(rows)
depth_table

In [ ]:
plot_df = depth_table[depth_table['max_depth'] != 'none']
plt.figure(figsize=(9, 5))
plt.plot(plot_df['max_depth'], plot_df['train AUC'], 'o-', label='train')
plt.plot(plot_df['max_depth'], plot_df['test AUC'],  'o-', label='test')
plt.xlabel('max_depth'); plt.ylabel('ROC-AUC')
plt.title('The classic overfitting picture')
plt.legend(); plt.show()

best = plot_df.loc[plot_df['test AUC'].idxmax()]
print(f"Best test AUC at max_depth={best['max_depth']}: {best['test AUC']:.4f}")

Train AUC climbs towards 1.0 and never stops. Test AUC rises, peaks somewhere around
depth 4-6, then **falls**. The gap between the two lines is the model memorising.

An unrestricted tree keeps splitting until every leaf is pure — in the limit, one leaf
per training row, which is a lookup table with extra steps.

**The knobs that stop this** (all of them are just "make leaves harder to create"):

| Parameter | What it does | Typical values |
|---|---|---|
| `max_depth` | hard cap on question count | 3-12 |
| `min_samples_leaf` | a leaf must hold this many rows | 1, 5, 20, 50, 100 |
| `min_samples_split` | a node needs this many rows to split at all | 2, 10, 50 |
| `max_features` | only consider a random subset of features per split | `'sqrt'`, 0.5, `None` |
| `ccp_alpha` | cost-complexity pruning: grow fully, then prune back | 0.0, 1e-4, 1e-3 |

You do not guess these. You search them — that is this afternoon's `GridSearchCV`.

### Checkpoint — answer before moving on

1. Why does a tree not need `StandardScaler`, when KNN badly does?
2. The unlimited tree got a near-perfect train AUC. Why is that *not* good news?
3. Which do you expect to overfit harder: `max_depth=20` with `min_samples_leaf=1`,
   or `max_depth=20` with `min_samples_leaf=100`? Why?

*(3: the second is far safer. Depth only caps how many questions you may ask; it is
`min_samples_leaf` that forbids a leaf built from three people.)*

---
# Phase 2 — Bagging and random forests (35 min)

## The problem we are solving

A deep tree has **low bias and high variance**. In plain English: it is flexible enough
to fit almost anything (low bias), but the exact tree you get depends enormously on
which rows happened to land in your training set (high variance).

Let's prove the variance claim rather than assert it. We train five deep trees on five
**bootstrap samples** — same size as the original, drawn with replacement, so each tree
sees roughly 63% of the unique rows plus duplicates — and then ask all five about the
same handful of people.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
n = len(X_train)
bootstrap_trees = []

for i in range(5):
    idx = rng.choice(n, size=n, replace=True)
    t = DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE)
    t.fit(X_train.iloc[idx], y_train.iloc[idx])
    bootstrap_trees.append(t)

sample = X_test.head(8)
preds = pd.DataFrame(
    {f'tree {i+1}': t.predict_proba(sample)[:, 1] for i, t in enumerate(bootstrap_trees)},
    index=[f'person {i+1}' for i in range(len(sample))])
preds['mean'] = preds.mean(axis=1)
preds['truth'] = y_test.head(8).values
preds.round(2)

Same algorithm, same data source, five different answers — often 0.0 from one tree and
1.0 from another about the *same person*. That spread is variance, and it is the thing
that limits a single tree.

Now the trick. Individually those trees are noisy in different directions, because the
noise comes from their random training samples rather than from any real signal. Average
them and the noise partly cancels while the signal survives.

In [ ]:
individual = [roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]) for t in bootstrap_trees]
averaged   = np.mean([t.predict_proba(X_test)[:, 1] for t in bootstrap_trees], axis=0)

print('Individual tree test AUCs:', ', '.join(f'{a:.3f}' for a in individual))
print(f'Best single tree:          {max(individual):.3f}')
print(f'Average of the five:       {roc_auc_score(y_test, averaged):.3f}')

The average beats every single member. That is **bagging** (bootstrap aggregating),
and it is the entire idea:

> Build many high-variance, low-bias models on resampled data. Average their predictions.
> Bias stays roughly where it was; variance drops by roughly a factor of the number of
> models — **but only to the extent the models make *independent* mistakes.**

That final clause is the whole reason random forests exist.

## From bagging to a random forest

Bootstrap samples overlap a lot, so bagged trees end up looking similar. If one feature
is strongly predictive — `GenHlth` here — nearly every tree picks it as the root split
and the trees become clones. Correlated mistakes do not cancel.

A **random forest** adds one line to the recipe: *at every split, only consider a random
subset of the features* (`max_features`). Sometimes `GenHlth` is not on the menu and the
tree is forced to find a different, weaker signal. The trees get individually worse and
collectively better.

Let's measure exactly that — how much the trees agree, and what it does to the score.

In [ ]:
def mean_pairwise_correlation(estimators, Xte):
    P = np.array([e.predict_proba(Xte)[:, 1] for e in estimators])
    corr = np.corrcoef(P)
    return corr[np.triu_indices_from(corr, k=1)].mean()

bagging = BaggingClassifier(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

forest = RandomForestClassifier(
    n_estimators=100, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

print(f'{"":26}{"tree agreement":>16}{"test AUC":>11}')
print(f'{"Bagged trees (all feats)":26}'
      f'{mean_pairwise_correlation(bagging.estimators_, X_test):>16.3f}'
      f'{roc_auc_score(y_test, bagging.predict_proba(X_test)[:, 1]):>11.4f}')
print(f'{"Random forest (sqrt)":26}'
      f'{mean_pairwise_correlation(forest.estimators_, X_test):>16.3f}'
      f'{roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]):>11.4f}')

Lower agreement, higher score. The forest's trees are *deliberately handicapped* and
that is the feature, not the bug.

> **Instructor note.** This is the one-sentence exam answer for "what is the difference
> between bagging and a random forest": bagging randomises the **rows**, a random forest
> randomises the **rows and the features at every split**, which decorrelates the trees.

## How many trees?

The single most common misconception about forests: that `n_estimators` is a knob you
can overfit with. It is not. More trees means a better estimate of the same average.
The curve rises and then flattens; it does not come back down. `n_estimators` is a
**budget** decision, not a **fit** decision.

In [ ]:
sizes = [1, 3, 5, 10, 25, 50, 100, 200, 400]
scores = []
big_forest = RandomForestClassifier(
    n_estimators=max(sizes), max_features='sqrt', random_state=RANDOM_STATE,
    n_jobs=-1).fit(X_train, y_train)

# Reuse one fitted forest: average the first k trees for each k.
tree_probs = np.array([t.predict_proba(X_test.values)[:, 1] for t in big_forest.estimators_])
for k in sizes:
    scores.append(roc_auc_score(y_test, tree_probs[:k].mean(axis=0)))

plt.figure(figsize=(8, 5))
plt.plot(sizes, scores, 'o-')
plt.xscale('log'); plt.xlabel('number of trees (log scale)'); plt.ylabel('test ROC-AUC')
plt.title('More trees never hurts accuracy — it just stops helping')
plt.show()
pd.DataFrame({'n_estimators': sizes, 'test AUC': np.round(scores, 4)})

The parameters that *do* control fit in a forest are the same ones that controlled the
single tree — `max_depth`, `min_samples_leaf`, `max_features`. Note that forests are
usually run with **fully grown** trees: the averaging is the regularisation.

## Out-of-bag scoring — a free validation set

Each tree skipped ~37% of the rows. Score every row using only the trees that never saw
it and you get a validation estimate without spending any data or doing any cross
validation. Useful when a fit is expensive.

In [ ]:
oob_forest = RandomForestClassifier(
    n_estimators=300, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1
).fit(X_train, y_train)

print(f'OOB accuracy:  {oob_forest.oob_score_:.4f}')
print(f'Test accuracy: {accuracy_score(y_test, oob_forest.predict(X_test)):.4f}')
print()
print('Reminder — always-predict-no baseline:', f'{1 - y_test.mean():.4f}')

> Two of those three numbers are nearly identical and the third is the reason we are not
> using accuracy today. A model that never predicts "diabetes" would score 0.86. Hold
> that thought until the project session.

## Which features mattered?

Forests hand you an importance ranking for free. They also hand you a well-known trap.

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(9, 7))
imp.plot.barh()
plt.title('Impurity-based feature importance (the default one)')
plt.xlabel('mean decrease in impurity'); plt.tight_layout(); plt.show()

**The trap.** `feature_importances_` measures how much each feature reduced impurity
during training. That statistic is biased towards features with **many distinct values**,
because a continuous feature gets thousands of candidate thresholds to get lucky with
while a 0/1 flag gets one. In this dataset `BMI` and `Age` have that advantage over
binary flags like `HighBP`, regardless of which actually matters.

The honest alternative is **permutation importance**: shuffle one column in the *test*
set, see how much the score drops. No training-time bias, and it answers the question
people actually mean — "how much does this model rely on this feature?"

In [ ]:
perm = permutation_importance(forest, X_test, y_test, n_repeats=5,
                              scoring='roc_auc', random_state=RANDOM_STATE, n_jobs=-1)

comparison = pd.DataFrame({
    'impurity (biased)': forest.feature_importances_,
    'permutation': perm.importances_mean,
}, index=X_train.columns)

comparison['impurity rank'] = comparison['impurity (biased)'].rank(ascending=False).astype(int)
comparison['permutation rank'] = comparison['permutation'].rank(ascending=False).astype(int)
comparison.sort_values('permutation', ascending=False).round(4)

Compare the two rank columns and find a feature that moves. That disagreement is the
bias, made visible.

> **Say this out loud in any stakeholder meeting:** importance means "this model leans on
> this column", never "this thing causes diabetes". Two correlated features will split
> the credit between them arbitrarily, and dropping one may not hurt the score at all.

### Checkpoint

1. Why does a random forest restrict features at each split when that makes each
   individual tree worse?
2. Your colleague says "I set `n_estimators=5000` and it overfitted." What went wrong
   with their diagnosis?
3. When would you prefer OOB score over 5-fold cross validation?

---
# Phase 3 — Boosting (35 min)

## A different idea entirely

Bagging builds many trees **in parallel** on different samples, then averages. Every
tree is an equal, independent vote.

Boosting builds trees **in sequence**, and each new tree is trained on *what the
previous ones got wrong*. The trees are not equal voters; they are a correction chain.

- Bagging fixes **variance** — many strong, unstable models, averaged into stability.
- Boosting fixes **bias** — many weak, stable models (shallow trees, often depth 3),
  stacked until together they are strong.

That is the whole distinction, and it is the most common interview question in this
area.

## Boosting by hand

Easiest to see on a regression toy. We fit a wiggly curve using nothing but depth-1
trees ("stumps"), one at a time, each fitting the leftover error of the last.

In [ ]:
rs = np.random.RandomState(0)
x_toy = np.sort(rs.uniform(-3, 3, 200))
y_toy = np.sin(x_toy) * 2 + rs.normal(0, 0.25, 200)
Xt = x_toy.reshape(-1, 1)

LEARNING_RATE = 0.5
prediction = np.full_like(y_toy, y_toy.mean())   # stage 0: predict the mean
stumps = []

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
snapshots = [1, 2, 3, 5, 20, 100]
snap_idx = 0

for step in range(1, max(snapshots) + 1):
    residual = y_toy - prediction                     # what we still get wrong
    stump = DecisionTreeRegressor(max_depth=1).fit(Xt, residual)
    prediction = prediction + LEARNING_RATE * stump.predict(Xt)
    stumps.append(stump)

    if step in snapshots:
        ax = axes.ravel()[snap_idx]; snap_idx += 1
        ax.scatter(x_toy, y_toy, s=10, alpha=0.4, label='data')
        ax.plot(x_toy, prediction, color='crimson', lw=2, label='model')
        mse = np.mean((y_toy - prediction) ** 2)
        ax.set_title(f'{step} stump(s) — MSE {mse:.3f}')
        ax.legend(fontsize=8)

plt.suptitle('Gradient boosting, built one stump at a time', fontsize=14)
plt.tight_layout(); plt.show()

That is gradient boosting, implemented in nine lines. Three things to notice:

1. **Every individual model is pathetic.** A depth-1 tree is a single step function. It
   could not fit a sine wave if its life depended on it. Together, 100 of them do.
2. **Each fits the residual**, not the target. Step 20 is not trying to predict `y`, it
   is trying to predict the little bit of `y` that steps 1-19 still miss.
3. **The learning rate shrinks every contribution.** Each stump's output is multiplied by
   0.5 before being added. Smaller steps, more of them, less chance of over-correcting
   on noise.

"Gradient" boosting because fitting the residual of squared error *is* taking a gradient
step on the loss. Swap the loss (log-loss for classification) and the same machinery
works — that is all `GradientBoostingClassifier` does.

## The learning rate / n_estimators trade-off

These two parameters are joined at the hip. Halve the learning rate and you need roughly
twice the trees to travel the same distance. Low-and-slow generalises better; it just
costs more compute.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, lr in zip(axes, [0.05, 0.5, 2.0]):
    pred = np.full_like(y_toy, y_toy.mean())
    for _ in range(30):
        pred = pred + lr * DecisionTreeRegressor(max_depth=1).fit(Xt, y_toy - pred).predict(Xt)
    ax.scatter(x_toy, y_toy, s=10, alpha=0.4)
    ax.plot(x_toy, pred, color='crimson', lw=2)
    ax.set_title(f'learning_rate={lr}, 30 stumps\nMSE {np.mean((y_toy - pred) ** 2):.3f}')
plt.tight_layout(); plt.show()

- `0.05` — under-shot. Still crawling towards the shape. Needs more rounds.
- `0.5` — about right for this budget.
- `2.0` — over-corrects on every round and chases noise. This is what a too-high
  learning rate looks like.

**Practical default:** `learning_rate=0.05` with a few hundred trees and early stopping.

## Boosting *can* overfit — unlike a forest

This is the sharpest practical difference between the two ensembles. Adding trees to a
forest plateaus harmlessly. Adding trees to a boosted model keeps chasing residuals, and
eventually the only residuals left are noise.

`staged_decision_function` lets us watch the score after every single round.

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.2, max_depth=4, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)

train_curve = [roc_auc_score(y_train, s) for s in gb.staged_decision_function(X_train)]
test_curve  = [roc_auc_score(y_test,  s) for s in gb.staged_decision_function(X_test)]

plt.figure(figsize=(9, 5))
plt.plot(train_curve, label='train')
plt.plot(test_curve, label='test')
best_round = int(np.argmax(test_curve))
plt.axvline(best_round, ls='--', color='grey')
plt.text(best_round + 8, min(test_curve), f'best at round {best_round + 1}')
plt.xlabel('boosting round'); plt.ylabel('ROC-AUC')
plt.title('Boosting overfits if you let it run too long')
plt.legend(); plt.show()

print(f'Best test AUC {max(test_curve):.4f} at round {best_round + 1}')
print(f'Test AUC after all 500 rounds: {test_curve[-1]:.4f}')

> **Instructor note.** Deliberately aggressive settings (`learning_rate=0.2`,
> `max_depth=4`) so the turn is visible in a live session. With `learning_rate=0.05` and
> stumps the curve is much flatter — which is exactly why those are the sane defaults.

The fix is **early stopping**: hold out a slice, watch its score every round, and stop
when it stops improving. You do not have to tune `n_estimators` at all — set it high and
let early stopping choose.

## The modern implementation

`GradientBoostingClassifier` is the original, and it is slow: exact split-finding over
every threshold, single-threaded. Modern boosters (`HistGradientBoostingClassifier`,
XGBoost, LightGBM, CatBoost) bin each feature into ~256 buckets first, which turns split
finding into a histogram scan. Same algorithm, 10-100x faster.

`HistGradientBoostingClassifier` ships with scikit-learn, needs no extra install,
handles NaNs natively, and does early stopping for you.

In [ ]:
import time

t0 = time.time()
GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X_train, y_train)
classic = time.time() - t0

t0 = time.time()
hgb = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.05,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
    random_state=RANDOM_STATE).fit(X_train, y_train)
modern = time.time() - t0

print(f'GradientBoostingClassifier (200 trees): {classic:.1f}s')
print(f'HistGradientBoosting (up to 500 + early stopping): {modern:.1f}s')
print(f'Early stopping chose {hgb.n_iter_} rounds out of 500.')
print(f'Test AUC: {roc_auc_score(y_test, hgb.predict_proba(X_test)[:, 1]):.4f}')

> **On XGBoost.** It is excellent and it is what you will meet in industry and on Kaggle.
> It is also `pip install xgboost` away, and its `XGBClassifier` follows the scikit-learn
> API exactly — `fit`, `predict_proba`, and it drops into `Pipeline` and `GridSearchCV`
> unchanged. If you have it installed, the cell below runs; if not, skip it. Nothing else
> in this course depends on it.

In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE)
    xgb.fit(X_train, y_train)
    print(f'XGBoost test AUC: {roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]):.4f}')
except ImportError:
    print('xgboost not installed — skipping. `pip install xgboost` if you want it.')

### Checkpoint

1. In one sentence each: what does bagging reduce, and what does boosting reduce?
2. You double `n_estimators` in a random forest and in a boosted model. What happens to
   test score in each case?
3. Why does a booster use shallow trees when a forest uses fully grown ones?

---
# Phase 4 — Which model, when (15 min)

## The honest bake-off

Everything we have built, on the same split, same metric.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lineup = {
    'Logistic regression': LogisticRegression(max_iter=1000),
    'Single tree (depth 5)': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Single tree (unlimited)': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=RANDOM_STATE, n_jobs=-1),
    'Hist gradient boosting': HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=400, early_stopping=True, random_state=RANDOM_STATE),
}

results = []
for name, mdl in lineup.items():
    cv_scores = cross_val_score(mdl, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    mdl.fit(X_train, y_train)
    results.append({
        'model': name,
        'CV AUC': cv_scores.mean(),
        'CV std': cv_scores.std(),
        'test AUC': roc_auc_score(y_test, mdl.predict_proba(X_test)[:, 1]),
    })

pd.DataFrame(results).set_index('model').round(4)

**Read the gaps, not the winner — and read the `CV std` column before you read anything
else.**

- The unlimited single tree is the worst thing in the table by a mile — genuinely worse
  than a straight line. Unpruned trees are not a model, they are a lookup table.
- Capping its depth recovers most of the loss.
- Ensembling beats every single tree by a clear margin.
- **And then the top three are tied.** Logistic regression, the forest, and the booster
  land inside a couple of thousandths of each other, while the fold-to-fold standard
  deviation is around 0.01 — four times the gap between them. Declaring a winner here
  would be reading noise.

That last point is the one to take away. On a dataset of mostly binary survey answers
with broadly additive effects, a linear model is genuinely competitive with the heavy
machinery. Ensembles pull ahead when there are strong interactions and non-linearities
to find; they are not a free upgrade on every problem. Boosting usually wins on tabular
data — *usually* is not *always*, and this table is what "not always" looks like.

> **Instructor note.** Resist the urge to rerun with a different seed until boosting
> wins. Sitting with a tie is the more valuable lesson, and it sets up the project
> session, where the decision that actually moves the needle turns out to be the
> **threshold**, not the algorithm.

## The one thing trees cannot do: extrapolate

A tree's prediction is always an average of training rows in some leaf. It therefore
**cannot output a value outside the range it was trained on.** Push it past the edge of
the training data and it flatlines.

In [ ]:
x_lin = np.linspace(0, 10, 100).reshape(-1, 1)
y_lin = 2 * x_lin.ravel() + 1 + np.random.RandomState(0).normal(0, 1, 100)
x_future = np.linspace(0, 20, 200).reshape(-1, 1)     # half of this is unseen territory

plt.figure(figsize=(9, 5))
plt.scatter(x_lin, y_lin, s=12, alpha=0.5, label='training data (x from 0 to 10)')
for name, mdl, style in [
        ('Linear regression', LinearRegression(), '-'),
        ('Random forest', RandomForestRegressor(n_estimators=100, random_state=0), '--')]:
    mdl.fit(x_lin, y_lin)
    plt.plot(x_future, mdl.predict(x_future), style, lw=2, label=name)
plt.axvline(10, color='grey', ls=':')
plt.text(10.3, 5, 'edge of training data', color='grey')
plt.legend(); plt.title('Trees cannot extrapolate. Lines can.')
plt.show()

The forest goes flat the instant it leaves familiar ground. If your problem involves
trend, time, or any quantity that genuinely keeps going — prices, growth, next year's
demand — that flat line is a serious liability and a linear model (or a hybrid) is the
better tool.

## The cheat sheet

| Situation | Reach for |
|---|---|
| Tabular data, mixed types, you want a strong result fast | **Gradient boosting** (HistGB / XGBoost / LightGBM) |
| Tabular, you want strong with almost no tuning | **Random forest** — defaults are hard to embarrass |
| Someone must read and sign off on the rules | **One shallow tree** — it is a flowchart |
| Relationship is genuinely linear, or you must extrapolate | **Linear / logistic regression** |
| You need calibrated probabilities out of the box | **Logistic regression** (or calibrate the ensemble) |
| Images, audio, text, sequences | **Neural networks** — that is Week 5 |
| Tiny dataset (< a few hundred rows) | Linear model. Ensembles need data to average over |

| | Single tree | Random forest | Boosting |
|---|---|---|---|
| Trees built | one | many, in parallel | many, in sequence |
| Each tree | deep | deep | shallow |
| Fixes | — | variance | bias |
| Overfits with more trees? | n/a | no | **yes** |
| Needs scaling? | no | no | no |
| Tuning effort | low | low | medium-high |
| Training speed | instant | fast (parallel) | slower (sequential) |
| Interpretable? | yes, fully | importances only | importances only |

## Universal truths about trees, worth memorising

1. **No scaling required.** Ever. For any tree-based model.
2. **Categoricals still need encoding** in scikit-learn — one-hot for low cardinality,
   ordinal for high. (`HistGradientBoostingClassifier` has native `categorical_features`
   support, and XGBoost/LightGBM handle it too.)
3. **Missing values**: `HistGradientBoosting`, XGBoost and LightGBM handle NaN natively.
   Plain `DecisionTreeClassifier` and `RandomForestClassifier` do not — impute first.
4. **Feature importance is not causation**, and the default one is biased. Use
   permutation importance when the answer matters.
5. **`n_estimators` is a budget for forests and a fit parameter for boosters.**

---
## Your turn

**1.** Re-run the depth sweep in Phase 1 with `min_samples_leaf=50` fixed. Does the
overfitting cliff move, flatten, or disappear?

**2.** In Phase 2, change the forest's `max_features` from `'sqrt'` to `1.0` (all
features). What happens to tree agreement, and to the test AUC? Explain the link.

**3.** Take the boosting curve from Phase 3 and re-run it at `learning_rate=0.02`.
Where does the test peak move to? Does the model still overfit within 500 rounds?

**4.** Build a forest using only the bottom five features by permutation importance.
How much worse is it than the full model? What does the size of that gap tell you about
how much of this dataset's signal lives in a handful of columns?

**5.** *Harder.* `HistGradientBoostingClassifier` accepts `class_weight='balanced'`.
Fit one with it and one without, and compare ROC-AUC **and** the confusion matrix at a
0.5 threshold. Explain why one metric barely moves and the other changes a lot.

---

**Next session:** the same dataset, the full 253,680 rows, `GridSearchCV` doing the
tuning properly — and then getting the resulting model out of the notebook and into a
Docker container alongside last week's regression model.